In [1]:
import torch
import torch.nn as nn
import math

In [2]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ffn_hidden_dim):
        super().__init__()
        # 1. Multi-Head Attention components
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_projection = nn.Linear(embed_dim, embed_dim)
        
        # 2. Normalization Layers
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        
        # 3. Position-wise Feed-Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ffn_hidden_dim),
            nn.ReLU(),
            nn.Linear(ffn_hidden_dim, embed_dim)
        )

    def split_heads(self, X):
        Batch_size, SeqLen, EmbedDim = X.shape
        return X.view(Batch_size, SeqLen, self.num_heads, self.head_dim).transpose(1, 2)

    def combine_heads(self, X):
        Batch_size, NumHeads, SeqLen, HeadDim = X.shape
        return X.transpose(1, 2).contiguous().view(Batch_size, SeqLen, NumHeads * HeadDim)

    def forward(self, x):
        # --- SUB-LAYER 1: Multi-Head Attention ---
        # Generate Q, K, V projections and split them into heads
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))
        
        # Scaled Dot-Product Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        weights = torch.softmax(scores, dim=-1)
        attention_out = torch.matmul(weights, V)
        
        # Re-stich heads and pass through the output project layer
        attention_out = self.combine_heads(attention_out)
        attention_out = self.out_projection(attention_out)
        
        # Residual Connection + LayerNorm 1
        x = self.norm1(x + attention_out)
        
        # --- SUB-LAYER 2: Feed-Forward Network ---
        ffn_out = self.ffn(x)
        
        # Residual Connection + LayerNorm 2
        x = self.norm2(x + ffn_out)
        
        return x

In [3]:
# Verify it runs smoothly
torch.manual_seed(42)
block = TransformerBlock(embed_dim=8, num_heads=2, ffn_hidden_dim=32)
sample_input = torch.randn(1, 3, 8) # Batch=1, Tokens=3, Features=8
sample_output = block(sample_input)

print("Block Input Shape: ", sample_input.shape)
print("Block Output Shape:", sample_output.shape)

Block Input Shape:  torch.Size([1, 3, 8])
Block Output Shape: torch.Size([1, 3, 8])


# Exercise

In [ ]:
import torch
import torch.nn as nn
import math

class CustomMultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        # Projection matrices
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        
        # Final output projection
        self.out_projection = nn.Linear(embed_dim, embed_dim, bias=False)
        
    def split_heads(self, x):
        B, S, E = x.shape
        return x.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        
    def combine_heads(self, x):
        B, H, S, D = x.shape
        return x.transpose(1, 2).contiguous().view(B, S, H * D)
        
    def forward(self, x):
        B, S, E = x.shape
        
        # 1. Project inputs to get Q, K, V tensors
        Q = nn.Linear(embed_dim, embed_dim)
        K = nn.Linear(embed_dim, embed_dim)
        V = nn.Linear(embed_dim, embed_dim)
        
        # 2. Split Q, K, V into multiple heads using self.split_heads
        Q_heads = self.split_head(self.W_q)
        K_heads = self.split_head(self.W_k)
        V_heads = self.split_head(self.W_v)
        
        # 3. Calculate raw attention scores (Q_heads times K_heads transposed)
        # Scaled by dividing by the square root of self.head_dim
        scores = (torch.matmul(Q_heads.transpose(1,2), K_heads.transpose(-2,-1)))/math.sqrt(self.head_dim)
        
        # 4. Apply softmax along the last dimension to get weights
        weights = torch.softmax(scores, dim=-1)
        
        # 5. Multiply weights by V_heads to get the final output context vectors
        attn_out = torch.matmul(weights, V)
        
        # 6. Recombine the heads back together using self.combine_heads
        combined = self.combine_heads(attn_out)
        
        # 7. Apply the final output linear projection layer
        output = self.out_projection(combined)
        
        return output

# --- Test Rig ---
torch.manual_seed(42)
mha = CustomMultiHeadAttention(embed_dim=8, num_heads=2)
test_input = torch.randn(1, 3, 8)  # Batch=1, SeqLen=3, EmbedDim=8
test_output = mha(test_input)

print("Final Output Shape:", test_output.shape)